# TenderScan — eTenders Scraper Prototype

This notebook demonstrates scraping public tender metadata from [eTenders.gov.za](https://www.etenders.gov.za).

> **Note:** We scrape public metadata only. No documents are downloaded or hosted. Data is attributed to the National Treasury eTender Portal.

## 1. Install Dependencies

In [ ]:
# Uncomment to install if needed
# !pip install requests beautifulsoup4 lxml pandas

## 2. Imports

In [ ]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

BASE_URL = "https://www.etenders.gov.za/Home/opportunities"
REQUEST_DELAY = 2  # seconds between requests

## 3. Fetch a Single Page

In [ ]:
def fetch_page(url, params=None):
    """Fetch a URL and return a BeautifulSoup object."""
    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        return BeautifulSoup(response.text, "lxml")
    except requests.RequestException as e:
        print(f"Error fetching {url}: {e}")
        return None

soup = fetch_page(BASE_URL)
print("Page title:", soup.title.string if soup else "Failed")

## 4. Parse Tender Listings

In [ ]:
def parse_tenders(soup):
    """Extract tender rows from the results table."""
    tenders = []
    if soup is None:
        return tenders
    rows = soup.select("table tbody tr")
    for row in rows:
        cells = row.find_all("td")
        if len(cells) < 5:
            continue
        tenders.append({
            "reference_number": cells[0].get_text(strip=True),
            "description": cells[1].get_text(strip=True),
            "advertised_by": cells[2].get_text(strip=True),
            "closing_date": cells[3].get_text(strip=True),
            "category": cells[4].get_text(strip=True),
            "source": "National Treasury eTender Portal",
            "scraped_at": datetime.utcnow().isoformat(),
        })
    return tenders

tenders = parse_tenders(soup)
print(f"Found {len(tenders)} tenders on page 1")

## 5. Scrape Multiple Pages

In [ ]:
all_tenders = []
MAX_PAGES = 3  # Adjust as needed

for page in range(1, MAX_PAGES + 1):
    print(f"Scraping page {page}...")
    s = fetch_page(BASE_URL, params={"page": page})
    page_tenders = parse_tenders(s)
    if not page_tenders:
        print("No tenders found, stopping.")
        break
    all_tenders.extend(page_tenders)
    time.sleep(REQUEST_DELAY)

print(f"Total tenders scraped: {len(all_tenders)}")

## 6. Explore the Data

In [ ]:
df = pd.DataFrame(all_tenders)
df.head(10)

## 7. Filter by Keyword

In [ ]:
KEYWORDS = ["ICT", "software", "digital"]
pattern = "|".join(KEYWORDS)
filtered = df[df["description"].str.contains(pattern, case=False, na=False)]
print(f"{len(filtered)} tenders match keywords: {KEYWORDS}")
filtered[["reference_number", "description", "closing_date"]]

## 8. Export to CSV

In [ ]:
import os

output_dir = os.path.join("..", "data")
os.makedirs(output_dir, exist_ok=True)
timestamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
output_path = os.path.join(output_dir, f"tenders_{timestamp}.csv")
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")